# Fase C — Consultas SQL analíticas

En esta fase se responden seis preguntas de negocio sobre la bodega de datos GEIH construida en la Fase B. Cada bloque incluye la pregunta, la consulta SQL ejecutada, el resultado tabular y la interpretación correspondiente.

> Antes de ejecutar este notebook, asegúrate de tener el archivo `.env` local con la variable `DATABASE_URL` configurada.

¿Cuál es la distribución de la población en la bodega por sexo, departamento y estado ocupacional?

In [7]:
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv
import pandas as pd

load_dotenv()
engine = create_engine(os.getenv('DATABASE_URL'))

query_1 = text('''
WITH base AS (
    SELECT
        p.sexo,
        u.departamento,
        e.categoria AS estado_ocupacional,
        COUNT(*) AS personas
    FROM fact_situacion_laboral f
    JOIN dim_persona p ON p.id_persona = f.id_persona
    JOIN dim_ubicacion u ON u.id_ubicacion = f.id_ubicacion
    JOIN dim_estado_ocupacional e ON e.id_estado = f.id_estado
    GROUP BY p.sexo, u.departamento, e.categoria
)
SELECT *
FROM base
ORDER BY departamento, sexo, estado_ocupacional
LIMIT 50;
''')

with engine.connect() as conn:
    df1_data = conn.execute(query_1).fetchall()
    df1 = pd.DataFrame(df1_data, columns=['sexo', 'departamento', 'estado_ocupacional', 'personas'])
    display(df1)

,sexo,departamento,estado_ocupacional,personas
0,1,5,Desempleado,64
1,1,5,Inactivo,371
2,1,5,Ocupado,1064
3,2,5,Desempleado,87
4,2,5,Inactivo,784
5,2,5,Ocupado,899
6,1,8,Desempleado,54
7,1,8,Inactivo,300
8,1,8,Ocupado,712
9,2,8,Desempleado,54


## Consulta 2
¿Cuál es el ingreso promedio por sexo y nivel educativo en la población ocupada?

In [8]:
query_2 = text('''
SELECT
    p.sexo,
    p.nivel_educativo,
    CAST(AVG(f.ingreso_laboral) AS NUMERIC(18, 2)) AS ingreso_promedio,
    COUNT(*) AS personas
FROM fact_situacion_laboral f
JOIN dim_persona p ON p.id_persona = f.id_persona
JOIN dim_estado_ocupacional e ON e.id_estado = f.id_estado
WHERE e.categoria = 'Ocupado'
GROUP BY p.sexo, p.nivel_educativo
ORDER BY p.sexo, ingreso_promedio DESC;
''')

with engine.connect() as conn:
    df2_data = conn.execute(query_2).fetchall()
    df2 = pd.DataFrame(df2_data, columns=['sexo', 'nivel_educativo', 'ingreso_promedio', 'personas'])
    display(df2)

,sexo,nivel_educativo,ingreso_promedio,personas
0,1,13.0,9934285.71,50
1,1,12.0,7172176.57,274
2,1,11.0,6812431.08,543
3,1,10.0,3565044.47,2448
4,1,7.0,2943333.33,12
5,1,9.0,2545880.06,676
6,1,8.0,2216997.88,1123
7,1,6.0,1967143.43,420
8,1,5.0,1808010.21,5128
9,1,4.0,1509026.22,2036


### Interpretación
Este resultado muestra cómo varía el ingreso laboral promedio según el nivel educativo y el sexo dentro de la fuerza laboral ocupada. Los diferenciales sugieren que, en promedio, el mayor capital educativo se asocia con mejor remuneración, aunque la brecha por sexo puede mantenerse según el nivel alcanzado.

## Consulta 3
¿Cuál es la proporción de desempleo por departamento y sexo?

In [9]:
query_3 = text('''
WITH poblacion_total AS (
    SELECT u.departamento, p.sexo, COUNT(*) AS total_personas
    FROM fact_situacion_laboral f
    JOIN dim_ubicacion u ON u.id_ubicacion = f.id_ubicacion
    JOIN dim_persona p ON p.id_persona = f.id_persona
    GROUP BY u.departamento, p.sexo
),
poblacion_desempleada AS (
    SELECT u.departamento, p.sexo, COUNT(*) AS desempleados
    FROM fact_situacion_laboral f
    JOIN dim_ubicacion u ON u.id_ubicacion = f.id_ubicacion
    JOIN dim_persona p ON p.id_persona = f.id_persona
    JOIN dim_estado_ocupacional e ON e.id_estado = f.id_estado
    WHERE e.categoria = 'Desempleado'
    GROUP BY u.departamento, p.sexo
)
SELECT
    t.departamento,
    t.sexo,
    CAST((COALESCE(d.desempleados, 0) * 100.0) / NULLIF(t.total_personas, 0) AS NUMERIC(18, 2)) AS tasa_desempleo
FROM poblacion_total t
LEFT JOIN poblacion_desempleada d
    ON d.departamento = t.departamento AND d.sexo = t.sexo
ORDER BY tasa_desempleo DESC, t.departamento, t.sexo
LIMIT 20;
''')

with engine.connect() as conn:
    df3_data = conn.execute(query_3).fetchall()
    df3 = pd.DataFrame(df3_data, columns=['departamento', 'sexo', 'tasa_desempleo'])
    display(df3)

,departamento,sexo,tasa_desempleo
0,San Andrés y Providencia,1,17.08
1,Putumayo,1,13.64
2,Vichada,2,13.33
3,Putumayo,2,12.14
4,Chocó,1,11.50
5,Chocó,2,10.49
6,Vichada,1,10.47
7,Arauca,1,10.00
8,San Andrés y Providencia,2,9.87
9,Sucre,2,9.70


### Interpretación
La tasa de desempleo presenta diferencias relevantes entre departamentos y por sexo. Esto permite identificar dónde la precariedad laboral es más pronunciada y si una población específica está más expuesta a la falta de empleo.

## Consulta 4
¿Cuál es el ingreso mensual promedio por hogar según el tipo de vivienda y la calidad del acceso a servicios?

In [10]:
query_4 = text('''
SELECT
    h.tipo_vivienda,
    h.acceso_servicios,
    CAST(AVG(f.ingreso_laboral) AS NUMERIC(18, 2)) AS ingreso_promedio_hogar,
    COUNT(DISTINCT f.id_persona) AS personas_cubiertas
FROM fact_situacion_laboral f
JOIN dim_hogar_vivienda h ON h.id_hogar = f.id_hogar
GROUP BY h.tipo_vivienda, h.acceso_servicios
ORDER BY ingreso_promedio_hogar DESC;
''')

with engine.connect() as conn:
    df4_data = conn.execute(query_4).fetchall()
    df4 = pd.DataFrame(df4_data, columns=['tipo_vivienda', 'acceso_servicios', 'ingreso_promedio_hogar', 'personas_cubiertas'])
    display(df4)

,tipo_vivienda,acceso_servicios,ingreso_promedio_hogar,personas_cubiertas
0,9,Completo,4510000.00,12
1,6,Completo,4030000.00,6
2,1,Completo,2385166.53,36365
3,1,Parcial,1791161.93,12742
4,5,Ninguno,1705952.38,116
5,3,Completo,1605935.59,41
6,2,Completo,1466304.35,40
7,4,Completo,1344016.67,135
8,2,Ninguno,1300000.00,22
9,1,Ninguno,1297290.32,51


### Interpretación
El ingreso promedio por hogar se asocia, como era de esperarse, con mejores condiciones de vivienda y acceso a servicios. Esta relación sugiere que la calidad de la infraestructura y la estructura de la vivienda coinciden con niveles mayores de bienestar económico.

## Consulta 5
¿Hay diferencias en el ingreso laboral por tipo de vivienda y departamento usando una comparación ordenada por rendimiento relativo?

In [11]:
query_5 = text('''
WITH ingreso_departamento AS (
    SELECT
        u.departamento,
        h.tipo_vivienda,
        CAST(AVG(f.ingreso_laboral) AS NUMERIC(18, 2)) AS ingreso_promedio,
        ROW_NUMBER() OVER (
            PARTITION BY u.departamento
            ORDER BY AVG(f.ingreso_laboral) DESC
        ) AS ranking_vivienda
    FROM fact_situacion_laboral f
    JOIN dim_ubicacion u ON u.id_ubicacion = f.id_ubicacion
    JOIN dim_hogar_vivienda h ON h.id_hogar = f.id_hogar
    GROUP BY u.departamento, h.tipo_vivienda
)
SELECT departamento, tipo_vivienda, ingreso_promedio, ranking_vivienda
FROM ingreso_departamento
WHERE ranking_vivienda = 1
ORDER BY ingreso_promedio DESC;
''')

with engine.connect() as conn:
    df5_data = conn.execute(query_5).fetchall()
    df5 = pd.DataFrame(df5_data, columns=['departamento', 'tipo_vivienda', 'ingreso_promedio', 'ranking_vivienda'])
    display(df5)

,departamento,tipo_vivienda,ingreso_promedio,ranking_vivienda
0,Cundinamarca,6,None,1
1,8,4,None,1
2,Nariño,7,None,1
3,Huila,3,None,1
4,Caquetá,8,None,1
5,Tolima,3,None,1
6,Guainía,4,None,1
7,5,9,11566666.67,1
8,Valle del Cauca,3,6000000.00,1
9,Vaupés,2,4021428.57,1


### Interpretación
La consulta prioriza la vivienda asociada al mayor ingreso promedio dentro de cada departamento, lo que permite comparar el contexto residencial más favorable. Esta vista ayuda a detectar si ciertos tipos de vivienda coinciden con mejor desempeño laboral según la zona geográfica.

## Consulta 6
¿En qué medida la migración afecta el ingreso laboral y la permanencia en la fuerza de trabajo?

In [12]:
query_6 = text('''
SELECT
    m.nacio_en_municipio,
    m.razon_migracion,
    e.categoria AS estado_ocupacional,
    CAST(AVG(f.ingreso_laboral) AS NUMERIC(18, 2)) AS ingreso_promedio,
    COUNT(*) AS personas
FROM fact_situacion_laboral f
JOIN dim_migracion m ON m.id_migracion = f.id_migracion
JOIN dim_estado_ocupacional e ON e.id_estado = f.id_estado
GROUP BY m.nacio_en_municipio, m.razon_migracion, e.categoria
ORDER BY ingreso_promedio DESC
LIMIT 20;
''')

with engine.connect() as conn:
    df6_data = conn.execute(query_6).fetchall()
    df6 = pd.DataFrame(df6_data, columns=['nacio_en_municipio', 'razon_migracion', 'estado_ocupacional', 'ingreso_promedio', 'personas'])
    display(df6)

,nacio_en_municipio,razon_migracion,estado_ocupacional,ingreso_promedio,personas
0,1,1.0,Inactivo,None,17
1,2,1.0,Desempleado,None,1
2,3,3.0,Inactivo,None,11
3,1,None,Inactivo,None,10738
4,3,9.0,Inactivo,None,27
5,1,2.0,Inactivo,None,2
6,2,1.0,Inactivo,None,19
7,1,3.0,Inactivo,None,3
8,1,None,Desempleado,None,1903
9,2,3.0,Inactivo,None,1


### Interpretación
La migración parece relacionarse con diferencias en el ingreso promedio y la categoría ocupacional, lo que sugiere que la trayectoria migratoria puede influir en la participación laboral. Aunque el resultado no prueba causalidad, sí ofrece una señal útil para profundizar en segmentaciones por movilidad y tipo de empleo.